# Mastering Keras Tuner: Automating Neural Network Architecture

Welcome to this hands-on guide! Our main goal here is **not** to build a perfectly accurate model, but rather to learn **how to use Keras Tuner**.

Because we are focusing strictly on the tuning workflow, we will skip standard data cleaning or preprocessing steps. We will feed raw data directly into the tuner to see how it automates the discovery of the best number of layers, neurons, and learning rates.# Mastering Keras Tuner: Automating Neural Network Architecture

Welcome to this hands-on guide! Our main goal here is **not** to build a perfectly accurate model, but rather to learn **how to use Keras Tuner**.

Because we are focusing strictly on the tuning workflow, we will skip standard data cleaning or preprocessing steps. We will feed raw data directly into the tuner to see how it automates the discovery of the best number of layers, neurons, and learning rates.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense
from keras.layers import Dropout

In [2]:
df =  pd.read_csv('diabetes.csv')

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [5]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

**We have splitted Training and Testing data**

Now before we build our model, we need to ensure Keras Tuner is installed in our environment and import the necessary deep learning libraries from TensorFlow...

***import keras_tuner as kt***

In [8]:
pip install keras-tuner -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 1.0 MB/s eta 0:00:00


In [12]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

---
### The Heart of the Tuner: The `build_model` Function

To use Keras Tuner, we don't just build a standard sequential model. Instead, we wrap our model inside a function. We pass a special `hp` (HyperParameters) object into this function, which allows us to define a **search space**.

in the code below.. watch how we use:
* `hp.Int()` to test different numbers of neurons.
* `hp.Choice()` to test different activation functions and learning rates.
* A `for` loop combined with `hp.Int()` to test a varying number of hidden layers!

In [13]:
from flax.nnx import optimizer
def build_model(hp):

  model = keras.Sequential()

  # No of hidden layers for our model
  num_layers = hp.Int('num_layers',min_value=1,max_value=10)

  for i in range(num_layers):
    model.add(Dense(
        units = hp.Int('unit_'+str(i),min_value = 8, max_value = 64, step = 8),

        activation = hp.Choice('activation',values = ['relu','tanh','elu'])
    ))

  model.add(Dense(1,activation='sigmoid'))

  optimizer = hp.Choice('optimizer',values = ['adam','sgd','rmsprop','adadelta'])

  learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

  model.compile(optimizer=optimizer,loss = 'binary_crossentropy',metrics=['accuracy'])

  return model


---
### Initializing the Random Search

Now we initialize the tuner. We will use `RandomSearch`, which randomly selects combinations of hyperparameters from our defined search space. We tell it to maximize our validation accuracy and limit it to 5 trials just to keep things fast for this tutorial.

### **What is a "Trial"? (The Architecture Selection)**

Trial is a single experiment with one unique combination of hyperparameters.
When you set max_trials=5 in your RandomSearch setup, you are telling the tuner:

**"*Pick 5 different random combinations of layers, neurons, and learning rates from our search space, and test them out*. "**


1.   Trial 1: 1 Layer, 16 Neurons, relu activation, 0.01 learning rate.
2.   Trial 2: 3 Layers, 64 Neurons, tanh activation, 0.001 learning rate.
3. ... and so on, up to 5 times.

**If you set max_trials too low (like 2 or 3), the tuner might
not stumble upon a good combination of neurons and layers because it didn't look far enough**

**If you set it higher (like 20 or 30), you give the random search a much better chance of finding a highly accurate architecture—but it will take significantly longer to finish running**


In [15]:
tuner = kt.RandomSearch(build_model,objective = 'val_accuracy',max_trials = 7,directory='tuner_logs',
    project_name='diabetes_tuning')

# to get a quick summary of search spacce we created
tuner.search_space_summary()

Search space summary
Default search space size: 5
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 1, 'max_value': 10, 'step': 1, 'sampling': 'linear'}
unit_0 (Int)
{'default': None, 'conditions': [], 'min_value': 8, 'max_value': 64, 'step': 8, 'sampling': 'linear'}
activation (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'tanh', 'elu'], 'ordered': False}
optimizer (Choice)
{'default': 'adam', 'conditions': [], 'values': ['adam', 'sgd', 'rmsprop', 'adadelta'], 'ordered': False}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}


### Executing the Search

Time to let Keras Tuner do the heavy lifting! We call `.search()`, passing in our training and validation data just like a standard `.fit()` method.

In [16]:
tuner.search(X_train,y_train,epochs=7,validation_data=(X_test,y_test))

Trial 7 Complete [00h 00m 07s]
val_accuracy: 0.6363636255264282

Best val_accuracy So Far: 0.6883116960525513
Total elapsed time: 00h 00m 39s


### Retrieving the Best Hyperparameters

Once the search finishes, we can easily grab the top-performing hyperparameters and print them out to see exactly what architecture worked best on our raw data.

In [17]:
Best_hp = tuner.get_best_hyperparameters()[0].values

In [18]:
Best_model = tuner.get_best_models()[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 12 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [21]:
print(f"{Best_hp}")

{'num_layers': 4, 'unit_0': 56, 'activation': 'tanh', 'optimizer': 'rmsprop', 'learning_rate': 0.01, 'unit_1': 32, 'unit_2': 32, 'unit_3': 16, 'unit_4': 56, 'unit_5': 32, 'unit_6': 32, 'unit_7': 40, 'unit_8': 48, 'unit_9': 8}


Now you can train you model easily....

In [22]:
Best_model.fit(X_train,y_train,epochs=100,validation_data=(X_test,y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.6873 - loss: 0.5845 - val_accuracy: 0.6494 - val_loss: 0.6683
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7068 - loss: 0.5778 - val_accuracy: 0.6623 - val_loss: 0.6517
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7134 - loss: 0.5617 - val_accuracy: 0.6948 - val_loss: 0.6414
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7117 - loss: 0.5569 - val_accuracy: 0.6753 - val_loss: 0.6356
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7085 - loss: 0.5661 - val_accuracy: 0.6818 - val_loss: 0.6224
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7264 - loss: 0.5480 - val_accuracy: 0.6688 - val_loss: 0.6221
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7150 - loss: 0.5457 - val_accuracy: 0.6948 - val_loss: 0.6159
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7182 - loss: 0.5422 - val_accuracy: 0.65

---
## Final Verdict: Is this the highest accuracy possible?

**Are you worried by the model's poor performance?**

Relax! Take a deep breath. The accuracy we achieved here is absolutely *not* the ceiling for this dataset.

If you are looking at the final score and wondering why the model isn't predicting diabetes with near-perfect accuracy, remember our primary objective: **We built this notebook to learn the mechanics of Keras Tuner, not to deploy a medical-grade AI.**

Here is exactly why our accuracy is currently limited and how you can improve it when you build your own models:

1. **Zero Data Preprocessing:** We intentionally skipped data cleaning and fed raw, unscaled data directly into the neural network. Deep learning models are highly sensitive to unscaled numbers. Adding a simple `StandardScaler` from Scikit-Learn before training would likely cause a massive jump in performance.(Dont feed gabage to your models 😉)

2. **Restricted Search Limits:** To keep this notebook running fast, we only allowed the tuner to run 5 trials and severely restricted the maximum number of layers and neurons. The "perfect" architecture might require 5 layers or 128 neurons, which our tuner wasn't even allowed to test!. You can increase the numbers to get better or best reults.

3. **Basic Search Algorithms:** We used `RandomSearch`, which guesses blindly. Upgrading to smarter algorithms built into the library, like `Hyperband` or `BayesianOptimization`, would allow the tuner to learn from its past trials and find better configurations faster.

### The Takeaway
You now have the blueprint for automating hyperparameter tuning! The absolute best way to master this is by doing. Take this workflow, apply it to a fresh project, add some proper data scaling, expand the search space, and watch your accuracy soar.

Happy coding!

by - PAXTON